In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
from helpers.pathlib import get_path_site_checkpoint
from IPython.display import display

from ata_pipeline1.helpers.enums import FieldNew, FieldSnowplow
from ata_pipeline1.site import AFRO_LA, DALLAS_FREE_PRESS, OPEN_VALLEJO, THE_19TH

plt.rcdefaults()
plt.style.use("ggplot")

In [ ]:
CSV_PREFIX = "221103-230125"

In [ ]:
# Read data from CHECKPOINT 7
df_afla = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 7, AFRO_LA.name))
df_dfp = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 7, DALLAS_FREE_PRESS.name))
df_ov = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 7, OPEN_VALLEJO.name))
df_19 = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 7, THE_19TH.name))

In [ ]:
df = pd.concat([df_afla, df_dfp, df_ov, df_19])

## Preliminary questions

### How many page views does each partner have per day?

In [ ]:
# Distribution of number of page views per day, for each partner
def describe_and_plot_page_views_per_day(df, start="2022-11-03", end="2023-01-25"):
    grouped_by_day = (
        df.groupby([FieldNew.SITE_NAME, df[FieldSnowplow.DERIVED_TSTAMP].dt.date])
        .size()
        .unstack(level=FieldSnowplow.DERIVED_TSTAMP)
        .loc[:, pd.date_range(start, end)]
        .fillna(0)
        .stack()
    )

    # Summary table
    display(grouped_by_day.groupby(FieldNew.SITE_NAME).apply(lambda x: pd.DataFrame(x.describe()).T).droplevel(-1))

    # Plot
    ax = grouped_by_day.unstack(level=FieldNew.SITE_NAME).plot(
        kind="bar", subplots=True, figsize=(12, 12), xlabel="Date (tick labels are Mondays)", ylabel="No. page views"
    )
    ax[-1].xaxis.set_major_locator(mdates.DayLocator(interval=7))
    plt.gcf().autofmt_xdate()
    plt.tight_layout()
    plt.show()


describe_and_plot_page_views_per_day(df)

## Fly-by users

In [ ]:
# Distribution of total dwell time per user, for each site
def describe_and_plot_dwell_time_per_user(df):
    df = df.query(f"{FieldNew.DWELL_SECS} != 0")

    grouped = df.groupby([FieldNew.SITE_NAME, FieldSnowplow.DOMAIN_USERID])[FieldNew.DWELL_SECS].sum().sort_values()

    # Summary table
    display(grouped.groupby(FieldNew.SITE_NAME).apply(lambda x: pd.DataFrame(x.describe()).T).droplevel(-1))

    # Box plots
    ax = grouped.unstack(level=FieldNew.SITE_NAME).plot(
        kind="box",
        logy=True,
        ylabel="Total dwell time per user"
        #             yticks=[0.01, 1, 100, 10000]
        #             subplots=True
    )
    ax.set_yticks([0.01, 1, 100, 10000])
    ax.set_yticklabels([0.01, 1, 100, 10000])
    plt.tight_layout()
    plt.show()


describe_and_plot_dwell_time_per_user(df)

## Fly-by subscribers

In [ ]:
# DFP subscribers and which page view in which session (visit) each of them subscribes


def get_subscribers_df(df):
    df_sub = (
        df.query(f"{FieldNew.FORM_SUBMIT_IS_NEWSLETTER} == True")[
            [
                FieldSnowplow.DERIVED_TSTAMP,
                FieldSnowplow.DOMAIN_USERID,
                FieldSnowplow.DOMAIN_SESSIONIDX,
                FieldNew.DOMAIN_SESSION_EVENTIDX,
                FieldSnowplow.PAGE_URLPATH,
                FieldSnowplow.REFR_URLHOST,
                #             FieldSnowplow.REFR_URLPATH
                FieldSnowplow.PAGE_REFERRER,
            ]
        ]
        .set_index(FieldSnowplow.DOMAIN_USERID)
        .sort_values([FieldSnowplow.DOMAIN_SESSIONIDX, FieldNew.DOMAIN_SESSION_EVENTIDX, FieldSnowplow.DERIVED_TSTAMP])
    )
    print(f"{df_sub.shape[0]:,} total subscriptions")
    print("Distribution of (session index, event index):")
    display(
        pd.DataFrame(
            df_sub.groupby([FieldSnowplow.DOMAIN_SESSIONIDX, FieldNew.DOMAIN_SESSION_EVENTIDX])
            .size()
            .rename("No. subscriptions")
        )
    )
    return df_sub


df_sub_dfp = get_subscribers_df(df_dfp)

In [ ]:
df_sub_ov = get_subscribers_df(df_ov)

In [ ]:
df_sub_19 = get_subscribers_df(df_19)

In [ ]:
df_sub_dfp

In [ ]:
df_sub_ov

In [ ]:
pd.options.display.max_rows = 10000
df_sub_19

In [ ]:
df_sub_19.shape

In [ ]:
df_sub_19[df_sub_19[FieldSnowplow.REFR_URLHOST].replace([None], [pd.NA]).isna()].shape

In [ ]:
df_sub_19[~df_sub_19[FieldSnowplow.REFR_URLHOST].replace([None], [pd.NA]).isna()]